# Day 3.10 — Pivotal Exercise: Compact Conversation History

This is the day's one hands-on implementation lab. It uses no API key. Try the starter cell first; a fully commented reference solution follows the check so you can compare or catch up.

## Why this mechanism matters

Conversation history grows without bound unless the application manages it. Compaction trades verbatim detail for a smaller representation, so its preservation rules must be explicit and testable.

## Contract

Return a new list without modifying the input. Preserve short histories unchanged. For longer histories, create one system summary containing older user facts and tool outcomes, followed by the most recent `keep_recent` messages.

Before coding, write one sentence predicting the easiest mistake to make.

In [ ]:
def compact_history(messages, keep_recent=2):
    """Return a compacted copy of messages.

    Short history (len <= keep_recent + 1): return an unchanged copy.
    Longer history: [ {"role": "system", "content": summary}, *last keep_recent messages ]
    The summary must mention older user facts and tool outcomes; assistant small talk may be dropped.
    """
    # TODO: never mutate the input list
    # TODO: return a copy when the history is already short
    # TODO: summarize older user and tool messages into one system message
    # TODO: keep the most recent messages verbatim
    raise NotImplementedError("Complete history compaction")

## Behavioural check

Run this after completing the starter cell. If you have not finished, it prints a hint instead of failing. A passing check proves the listed contract examples, not every possible input.

In [ ]:
def run_checks():
    history = [
        {"role": "user", "content": "My preferred unit is millimetres."},
        {"role": "assistant", "content": "Noted."},
        {"role": "tool", "content": "calculation completed: 25 mm"},
        {"role": "user", "content": "Use that result in the report."},
    ]
    compacted = compact_history(history, keep_recent=2)
    print("Before:", len(history), "messages   After:", len(compacted), "messages")
    for message in compacted:
        print(f"  {message['role']:>9}: {message['content']}")
    assert len(history) == 4 and history[0]["content"].startswith("My preferred"), "input must not be mutated"
    assert len(compacted) == 3
    assert compacted[0]["role"] == "system" and "millimetres" in compacted[0]["content"], "older user facts survive in the summary"
    assert compacted[-2:] == history[-2:], "recent messages stay verbatim"

    short = history[:2]
    kept = compact_history(short, keep_recent=2)
    assert kept == short and kept is not short, "short histories are returned as an unchanged copy"
    print("PASS: history is bounded, essential information is retained, input is untouched")

try:
    run_checks()
except NotImplementedError:
    print("Not implemented yet. Complete the starter cell above, or study the reference solution below and re-run this cell.")

## Reference solution

Read this even if your check passed: compare each commented line with your version, then re-run the check cell above.

In [ ]:
# --- Reference solution: read it line by line, then re-run the check cell above ---
def compact_history(messages, keep_recent=2):
    if len(messages) <= keep_recent + 1:          # nothing worth compacting
        return list(messages)                      # a copy, so the caller's list is untouched
    older, recent = messages[:-keep_recent], messages[-keep_recent:]
    facts = []
    for message in older:                          # decide what deserves to survive
        if message["role"] == "user":
            facts.append(f"user said: {message['content']}")
        elif message["role"] == "tool":
            facts.append(f"tool result: {message['content']}")
        # assistant acknowledgements such as "Noted." carry no facts and are dropped
    summary = {"role": "system", "content": "Summary of earlier conversation: " + " | ".join(facts)}
    return [summary, *recent]                      # one summary + the verbatim recent tail

print("Reference compact_history defined. Re-run the check cell above to see PASS.")

## Explain

**Which details are unsafe to summarize away?**

<details><summary>Show answer</summary>

Anything that still governs future behaviour: an unresolved approval request, an exact constraint (a deadline, a unit, a budget), a safety instruction, or a pending action id. Those belong in explicit application state, not only in a lossy summary.

</details>

**Where should an unresolved approval request live: summary, state, or both?**

<details><summary>Show answer</summary>

State, always: the approval boundary in Day 3.7 must be able to find the exact proposed action. The summary may mention it for the model's benefit, but the application must not rely on the summary to enforce it.

</details>